In [1]:
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: LOADING MESSY REAL-WORLD DATA
# ==========================================
# Creating a realistic messy dataset (just like the real Zomato Kaggle dataset)
raw_data = {
    'restaurant_name': ['Truffles', 'Empire Restaurant', 'Meghana Foods', 'Onesta', 'Chai Point', 
                        'Truffles', 'Corner House', 'Udupi Palace', 'Biryani Zone', 'Farzi Cafe',
                        'KFC', 'Pizza Hut', 'Local Dhaba', 'The Black Pearl', 'Rameshwaram Cafe'],
    'location': ['Koramangala', 'Indiranagar', 'Koramangala', 'HSR', 'Whitefield', 
                 'Koramangala', 'Indiranagar', 'HSR', 'Whitefield', 'UB City',
                 'MG Road', 'Koramangala', 'Yeshwantpur', 'Koramangala', 'Indiranagar'],
    'online_order': ['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes'],
    'book_table': ['No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No'],
    'rate': ['4.7/5', '4.1/5', '4.6/5', '4.3/5', 'NEW', '4.7/5', '4.6/5', '3.8/5', '-', '4.5/5', 
             '4.0/5', '3.9/5', np.nan, '4.7/5', '4.8/5'],
    'votes': [3500, 1800, 4200, 1500, 20, 3500, 2100, 400, 10, 1900, 1200, 850, 5, 3100, 5000],
    'approx_cost(for two people)': ['900', '750', '1,000', '800', '300', '900', '500', '250', '600', '2,200', 
                                    '650', '800', np.nan, '1,800', '400'],
    'rest_type': ['Cafe', 'Casual Dining', 'Casual Dining', 'Cafe', 'Quick Bites', 
                  'Cafe', 'Dessert Parlor', 'Quick Bites', 'Casual Dining', 'Fine Dining',
                  'Quick Bites', 'Casual Dining', 'Dhaba', 'Casual Dining', 'Quick Bites']
}

df = pd.DataFrame(raw_data)

print("--- INITIAL MESSY DATA ---")
print(df.head())
print("\nMissing values before cleaning:\n", df.isnull().sum())


# ==========================================
# STEP 2: DATA CLEANING (PANDAS & NUMPY)
# ==========================================

# 1. Remove duplicate rows (Truffles is repeated twice)
df = df.drop_duplicates().reset_index(drop=True)

# 2. Clean the 'rate' column (Remove '/5', replace 'NEW' and '-' with NaN, convert to float)
def clean_rating(value):
    if pd.isnull(value) or value == 'NEW' or value == '-':
        return np.nan
    else:
        return float(str(value).split('/')[0])

df['rate'] = df['rate'].apply(clean_rating)

# Fill missing ratings with the median rating using NumPy
df['rate'] = df['rate'].fillna(df['rate'].median())

# 3. Clean 'approx_cost(for two people)' (Remove commas like '1,000' -> 1000)
df['cost_for_two'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '')
df['cost_for_two'] = pd.to_numeric(df['cost_for_two'], errors='coerce')

# Fill missing cost with average (mean) cost
mean_cost = np.round(np.nanmean(df['cost_for_two']), 0)
df['cost_for_two'] = df['cost_for_two'].fillna(mean_cost)

# Drop the old messy column
df = df.drop(columns=['approx_cost(for two people)'])

print("\n--- CLEANED DATA ---")
print(df.info())


# ==========================================
# STEP 3: FEATURE ENGINEERING WITH NUMPY
# ==========================================
# Let's categorize restaurants into Budget, Mid-Range, and Luxury using np.select()
conditions = [
    (df['cost_for_two'] <= 400),
    (df['cost_for_two'] > 400) & (df['cost_for_two'] <= 1000),
    (df['cost_for_two'] > 1000)
]
choices = ['Budget', 'Mid-Range', 'Luxury']

df['price_category'] = np.select(conditions, choices, default='Mid-Range')


# ==========================================
# STEP 4: EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
print("\n--- KEY BUSINESS INSIGHTS ---")

# Insight 1: Statistical summary using NumPy
print(f"1. Average Cost for Two: ₹{np.mean(df['cost_for_two']):.2f}")
print(f"2. 90th Percentile Cost (Luxury Threshold): ₹{np.percentile(df['cost_for_two'], 90):.2f}")
print(f"3. Rating Standard Deviation: {np.std(df['rate']):.2f}")

# Insight 2: Does online order impact average ratings and votes? (Pandas GroupBy)
online_impact = df.groupby('online_order')[['rate', 'votes', 'cost_for_two']].mean().round(2)
print("\n4. Impact of Online Ordering on Ratings & Votes:")
print(online_impact)

# Insight 3: Top locations by restaurant count and average rating
location_summary = df.groupby('location').agg(
    total_restaurants=('restaurant_name', 'count'),
    avg_rating=('rate', 'mean'),
    avg_cost=('cost_for_two', 'mean')
).sort_values(by='total_restaurants', ascending=False).round(2)

print("\n5. Location-wise Restaurant Breakdown:")
print(location_summary)

# Save cleaned dataset for portfolio proof
df.to_csv('cleaned_zomato_data.csv', index=False)
print("\nCleaned data successfully saved as 'cleaned_zomato_data.csv'!")

--- INITIAL MESSY DATA ---
     restaurant_name     location online_order book_table   rate  votes  \
0           Truffles  Koramangala          Yes         No  4.7/5   3500   
1  Empire Restaurant  Indiranagar          Yes         No  4.1/5   1800   
2      Meghana Foods  Koramangala          Yes         No  4.6/5   4200   
3             Onesta          HSR          Yes        Yes  4.3/5   1500   
4         Chai Point   Whitefield          Yes         No    NEW     20   

  approx_cost(for two people)      rest_type  
0                         900           Cafe  
1                         750  Casual Dining  
2                       1,000  Casual Dining  
3                         800           Cafe  
4                         300    Quick Bites  

Missing values before cleaning:
 restaurant_name                0
location                       0
online_order                   0
book_table                     0
rate                           1
votes                          0
approx_c